# 📓 Actividad Extra — Crea tu videojuego con IA, usando HTML

**Complementa al Notebook 3 (Prototipo del videojuego)** · Se puede hacer en cualquier momento de la Sección 2 o 3

### Objetivo de hoy
Diseñar **prompts efectivos** para que una IA (Claude o Gemini) genere, en **HTML + CSS + JavaScript**, un videojuego funcional que represente su sistema ecológico-urbano — como una forma alternativa (o complementaria) de construir la interfaz, dirigiendo a una IA en vez de escribir cada línea ustedes mismos.

Aquí **el lenguaje de programación es HTML** (con su CSS y JavaScript), no Python. El notebook lo van a usar sobre todo para escribir sus prompts, guardar el código que les regrese la IA, y **previsualizar el juego directamente aquí**, sin salir de Jupyter/Colab.


## 0. Conectemos con lo anterior

En el Notebook 3 construyeron un prototipo de videojuego escribiendo Python línea por línea, con ayuda de esta libreta como guía.

**✏️ Antes de empezar, respondan como equipo:**

1. ¿Qué parte de programar el prototipo del Notebook 3 les costó más trabajo?
2. Si en vez de escribir el código ustedes mismos, le **describieran** a una IA exactamente qué necesitan y ella escribiera el código, ¿qué creen que sería más fácil? ¿Qué creen que sería más difícil o riesgoso?
3. ¿Alguna vez han usado Claude, ChatGPT o Gemini para generar código? ¿Qué tan bien funcionó a la primera?


## 1. Mini-lección: ¿qué hace que un prompt para generar código sea bueno?

Pedirle a una IA "hazme un videojuego" casi nunca funciona bien. Un buen prompt para generar código tiene:

1. **Contexto**: qué representa el juego, para quién es (estudiantes sin experiencia en programación revisando el código después).
2. **Especificaciones técnicas exactas**: qué lenguaje, cuántos archivos, qué debe poder hacer el usuario (clic, botones, etc.).
3. **Restricciones claras**: por ejemplo, "todo en un solo archivo HTML autocontenido, sin librerías externas, para poder abrirlo con solo hacer doble clic".
4. **Estructura de datos concreta**: las variables y reglas de SU modelo (no genéricas) — entre más específico, menos tiene que inventar la IA por su cuenta.
5. **Pedir comentarios en el código**: para que ustedes puedan leerlo y entenderlo, no solo copiarlo.

**🤔 Antes de continuar:** de estos 5 elementos, ¿cuál creen que es el que más gente olvida poner en sus prompts? ¿Por qué creen que ese olvido causa que la IA "invente" cosas que no querían?


In [ ]:
# Ejecuta esta celda para preparar las herramientas que usaremos hoy.
import json
import os
import html as html_lib
from IPython.display import display, HTML

print("Listo ✅ — vamos a construir un prompt personalizado con los datos de tu equipo.")


## 2. Generar un prompt personalizado a partir de su propio modelo

Si ya tienen su `modelo_mental.json` (del Notebook 1) en esta misma carpeta, la siguiente celda arma automáticamente un prompt detallado usando SUS variables y relaciones reales — así no tienen que describirlas todas a mano.


In [ ]:
if os.path.exists("modelo_mental.json"):
    with open("modelo_mental.json", "r", encoding="utf-8") as f:
        modelo = json.load(f)
    print("✅ Modelo cargado desde modelo_mental.json")
else:
    print("⚠️ No se encontró modelo_mental.json — usando un modelo de ejemplo.")
    modelo = {
        "equipo": "Equipo de ejemplo",
        "zona_de_estudio": "Zona de ejemplo",
        "variables": [
            {"nombre": "vegetacion", "descripcion": "Cobertura vegetal", "unidad": "%", "rango": [0, 100], "valor_inicial": 60},
            {"nombre": "construccion", "descripcion": "Área construida", "unidad": "%", "rango": [0, 100], "valor_inicial": 30},
            {"nombre": "poblacion", "descripcion": "Habitantes (miles)", "unidad": "miles", "rango": [0, 500], "valor_inicial": 100},
            {"nombre": "contaminacion", "descripcion": "Nivel de contaminación", "unidad": "índice", "rango": [0, 100], "valor_inicial": 20},
            {"nombre": "agua", "descripcion": "Calidad/disponibilidad de agua", "unidad": "índice", "rango": [0, 100], "valor_inicial": 70},
        ],
        "relaciones": [
            {"origen": "construccion", "destino": "vegetacion", "signo": -1, "fuerza": 0.4, "explicacion": "Más construcción reduce vegetación"},
            {"origen": "poblacion", "destino": "contaminacion", "signo": 1, "fuerza": 0.3, "explicacion": "Más población, más contaminación"},
        ],
    }

variables_texto = "\n".join(
    f"- {v['nombre']} ({v['descripcion']}): rango {v['rango']}, valor inicial {v['valor_inicial']}"
    for v in modelo["variables"]
)
relaciones_texto = "\n".join(
    f"- {r['origen']} afecta a {r['destino']} ({'aumenta' if r['signo'] > 0 else 'disminuye'}, fuerza {r['fuerza']})"
    for r in modelo["relaciones"]
)

prompt_generado = f"""Eres un asistente de programación ayudando a estudiantes de secundaria SIN experiencia previa en programación.

Quiero que generes un videojuego educativo COMPLETO en un solo archivo HTML autocontenido (HTML + CSS + JavaScript, todo en el mismo archivo, sin librerías externas ni CDNs), que se pueda abrir directamente en un navegador con doble clic.

CONTEXTO: el juego representa el sistema ecológico-urbano de la zona "{modelo['zona_de_estudio']}" (equipo {modelo['equipo']}).

VARIABLES DEL SISTEMA (deben verse como indicadores numéricos en pantalla, actualizándose en cada turno):
{variables_texto}

RELACIONES ENTRE VARIABLES (usa estas reglas simples para actualizar las variables en cada turno):
{relaciones_texto}

REQUISITOS DEL JUEGO:
1. Un tablero tipo cuadrícula (grid) de 10x10 celdas dibujado con HTML/CSS (puede ser con <div> o <canvas>), donde cada celda representa un tipo de uso de suelo (agua, vegetación, urbano, industrial) con un color distinto.
2. Al hacer clic en una celda, debe cambiar de tipo (cicla entre las 4 opciones).
3. Un botón "Siguiente turno" que recalcule las variables del sistema según las reglas de arriba y actualice los indicadores en pantalla.
4. Los indicadores deben mostrarse siempre visibles, con su valor numérico actualizado.
5. Comenta el código en español, explicando qué hace cada parte, porque lo van a leer estudiantes principiantes.
6. Que el diseño visual sea simple pero agradable (colores, espaciado, tipografía legible).

Genera el archivo HTML completo, listo para copiar y pegar."""

print(prompt_generado)


### 🔍 Antes de enviar el prompt — reflexionen

**✏️ Escriban su respuesta:**

- Lean el prompt generado arriba. ¿Le falta algo específico de SU modelo que la IA necesitaría saber? Edítenlo si hace falta antes de usarlo. *(reemplazar)*
- **Predicción:** en una escala del 1 al 5, ¿qué tan completo y correcto creen que será el HTML que genere la IA en su primer intento? ¿Qué parte creen que le va a faltar o va a fallar? *(reemplazar)*


## 3. Usar el prompt con Claude o Gemini

1. Copien el prompt de la celda de arriba (la salida de texto, no el código Python).
2. Abran [claude.ai](https://claude.ai) o [gemini.google.com](https://gemini.google.com) en una nueva pestaña.
3. Peguen el prompt y envíenlo.
4. Copien **todo** el código HTML que les regrese la IA (debe empezar con `<!DOCTYPE html>` y terminar con `</html>`).
5. Peguen ese código en la celda de la Sección 4, reemplazando el ejemplo que ya está ahí.

> 💡 Si la respuesta de la IA viene con explicaciones antes o después del código, solo copien el bloque de código (normalmente aparece en un recuadro gris con fondo oscuro).


## 4. Guardar y previsualizar su juego

La celda de abajo ya trae un **juego base de ejemplo** (funciona sin necesidad de la IA, para que puedan ver cómo se previsualiza). **Reemplacen el contenido de `codigo_html_ia` con el código que les dio Claude o Gemini**, y vuelvan a correr la celda para ver su propio juego.


In [ ]:
# ✏️ REEMPLAZA todo el contenido entre las comillas triples con el código HTML que te dio la IA.
# Este ejemplo de base funciona por sí solo para que veas cómo se previsualiza.

codigo_html_ia = """<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<title>Mi Ciudad — Ejemplo base</title>
<style>
  body { font-family: Arial, sans-serif; text-align: center; background: #f4f4f0; }
  #tablero { display: grid; grid-template-columns: repeat(10, 32px); gap: 2px; justify-content: center; margin: 20px auto; }
  .celda { width: 32px; height: 32px; cursor: pointer; border-radius: 3px; }
  #indicadores { margin: 12px; font-size: 15px; }
  button { padding: 8px 16px; font-size: 14px; border-radius: 6px; border: none; background: #2e7d4f; color: white; cursor: pointer; }
</style>
</head>
<body>
  <h2>🏙️ Mi Ciudad — Ejemplo base (reemplaza esto con el código de la IA)</h2>
  <div id="indicadores">
    Vegetación: <span id="ind-veg">50</span>% |
    Contaminación: <span id="ind-cont">20</span>% |
    Turno: <span id="ind-turno">0</span>
  </div>
  <div id="tablero"></div>
  <button onclick="siguienteTurno()">Siguiente turno</button>

<script>
// Tipos de celda: 0 = agua, 1 = vegetacion, 2 = urbano, 3 = industrial
const colores = ["#3b82c4", "#4caf6b", "#9e9e9e", "#e08a2c"];
const N = 10;
let grid = [];
let turno = 0;
let vegetacion = 50;
let contaminacion = 20;

// Crear grid inicial aleatorio (más probabilidad de vegetacion y urbano)
for (let i = 0; i < N * N; i++) {
  const r = Math.random();
  grid.push(r < 0.5 ? 1 : (r < 0.8 ? 2 : (r < 0.9 ? 3 : 0)));
}

function dibujarTablero() {
  const tablero = document.getElementById("tablero");
  tablero.innerHTML = "";
  grid.forEach((tipo, idx) => {
    const div = document.createElement("div");
    div.className = "celda";
    div.style.background = colores[tipo];
    div.onclick = () => {
      grid[idx] = (grid[idx] + 1) % 4;  // ciclar tipo de celda
      dibujarTablero();
    };
    tablero.appendChild(div);
  });
}

function siguienteTurno() {
  turno += 1;
  const pctVeg = 100 * grid.filter(t => t === 1).length / grid.length;
  const pctUrbanoIndustrial = 100 * grid.filter(t => t === 2 || t === 3).length / grid.length;
  vegetacion = pctVeg;
  contaminacion = Math.min(100, Math.max(0, contaminacion + 0.3 * pctUrbanoIndustrial - 0.2 * pctVeg));

  document.getElementById("ind-veg").innerText = vegetacion.toFixed(1);
  document.getElementById("ind-cont").innerText = contaminacion.toFixed(1);
  document.getElementById("ind-turno").innerText = turno;
}

dibujarTablero();
</script>
</body>
</html>
"""

with open("mi_juego.html", "w", encoding="utf-8") as f:
    f.write(codigo_html_ia)

print("✅ Guardado como 'mi_juego.html' — corre la siguiente celda para previsualizarlo aquí mismo.")


In [ ]:
# Previsualización dentro del notebook (funciona en Jupyter y en Colab)

html_escapado = html_lib.escape(codigo_html_ia)
display(HTML(f'<iframe srcdoc="{html_escapado}" width="100%" height="480" style="border:1px solid #ccc; border-radius:8px;"></iframe>'))


### 🔍 Observen y reflexionen

Prueben su juego en la previsualización de arriba: hagan clic en algunas celdas y en "Siguiente turno".

**✏️ Comparen el resultado con su predicción de la Sección 2. Escriban su respuesta:**

- ¿Qué tan completo/correcto estuvo el código que les dio la IA en su primer intento? ¿Coincide con lo que predijeron? *(reemplazar)*
- ¿Qué tuvieron que corregir o pedir de nuevo? *(reemplazar)*
- ¿El juego generado realmente refleja las variables y relaciones de SU modelo, o la IA "inventó" algo genérico? *(reemplazar)*


## 5. Iterar: pedirle mejoras y correcciones a la IA

Rara vez el primer resultado es perfecto — la clave de trabajar con IA es **iterar**. Algunos prompts útiles para la siguiente ronda:

1. *"Este es mi código HTML actual: [pegar código]. Cuando hago clic en una celda no pasa nada, en vez de cambiar de color. ¿Qué está mal?"*
2. *"Agrega un tercer indicador llamado 'economía' que suba cuando hay más celdas industriales y urbanas, usando el mismo estilo de los indicadores que ya existen."*
3. *"El tablero se ve muy apretado en pantallas pequeñas. Ajusta el CSS para que las celdas sean un poco más grandes y se vea bien en celular."*
4. *"Explícame, línea por línea, qué hace la función `siguienteTurno()` de este código, como si nunca hubiera programado en JavaScript."*
5. *"Agrega un mensaje que aparezca cuando la contaminación llegue a 100, diciendo que la ciudad colapsó."*

**Regla de oro (la misma de los Notebooks 2 y 4):** lean y entiendan cada cambio que la IA hace antes de aceptarlo — el objetivo es que ustedes puedan explicar su propio juego, no solo mostrarlo.


## 6. Registro de prompts (documenten su proceso)

Llevar un registro de qué le pidieron a la IA y qué cambió es parte de un buen proceso de trabajo con IA — y les servirá para explicar su proceso en la presentación final.

**✏️ Completen esta tabla con al menos 3 prompts que usaron:**

| # | Prompt que escribieron (resumen) | Qué cambió en el juego | ¿Lo aceptaron tal cual o lo ajustaron? |
|---|---|---|---|
| 1 | *(reemplazar)* | *(reemplazar)* | *(reemplazar)* |
| 2 | *(reemplazar)* | *(reemplazar)* | *(reemplazar)* |
| 3 | *(reemplazar)* | *(reemplazar)* | *(reemplazar)* |


## 🎨 Ejercicio: conecten su juego con los indicadores del Notebook 4

Pídanle a la IA que agregue los mismos 3 indicadores que usaron en el Notebook 4 (calidad de vida, biodiversidad, economía), calculados con una fórmula parecida a la de `calcular_indicadores()`. Peguen el prompt que usaron y el resultado en la Sección 6.


## 🧭 Bitácora de aprendizaje

**✏️ Completen cada punto como equipo:**

1. Comparen esta actividad con el Notebook 3: ¿qué fue más rápido — programar ustedes mismos en Python, o dirigir a la IA en HTML? ¿Y qué entendieron mejor? *(reemplazar)*
2. Aunque la IA haya escrito el código, ¿qué decisiones siguieron siendo 100% suyas como equipo? *(reemplazar)*
3. ¿Cuál creen que es el riesgo de usar código generado por IA sin leerlo ni entenderlo? Den un ejemplo concreto de algo que podría salir mal. *(reemplazar)*
4. Si tuvieran que enseñarle a otro equipo a escribir un buen prompt para generar código, ¿qué 2 consejos le darían? *(reemplazar)*

## ✅ Producto esperado de esta actividad

- [ ] Prompt personalizado generado y enviado a Claude o Gemini
- [ ] Archivo `mi_juego.html` funcional, generado (o mejorado) con ayuda de la IA
- [ ] Previsualización del juego corriendo dentro del notebook
- [ ] Registro de al menos 3 prompts documentados
- [ ] Bitácora de aprendizaje completa

Este archivo `mi_juego.html` puede complementar (o servir de alternativa) al prototipo de Pygame/`ipywidgets` del Notebook 3 dentro de su proyecto final. 🎮
